In [2]:
import requests
import zipfile
import os
import glob
import cv2
import matplotlib.pyplot as plt
import random

In [2]:
def download_file(url, save_name):
    url = url
    if not os.path.exists(save_name):
        file = requests.get(url)
        open(save_name, 'wb').write(file.content)

In [3]:
def unzip(zip_file=None):
    try:
        with zipfile.ZipFile(zip_file) as z:
            z.extractall("./")
            print("Extracted all")
    except:
        print("Invalid file")

In [5]:
download_file(
    'https://www.dropbox.com/s/ievh0sesad015z0/trash_inst_material.zip?dl=1',
    'trash_inst_material.zip'
)

unzip(zip_file='trash_inst_material.zip')

Extracted all


In [3]:
cwd = os.getcwd()
print(cwd)

d:\Projects\Capstone Project\model switching


In [4]:
import yaml

attr = {
    'path': cwd+'/trash_inst_material',
    'train': 'train/images',
    'val': 'val/images',

    'names': {
        0: 'rov',
        1: 'plant',
        2: 'animal_fish',
        3: 'animal_starfish',
        4: 'animal_shells',
        5: 'animal_crab',
        6: 'animal_eel',
        7: 'animal_etc',
        8: 'trash_etc',
        9: 'trash_fabric',
        10: 'trash_fishing_gear',
        11: 'trash_metal',
        12: 'trash_paper',
        13: 'trash_plastic',
        14: 'trash_rubber',
        15: 'trash_wood',
    }
}

In [5]:
with open('trashcan_inst_material.yaml', 'w') as f:
    yaml.dump(attr, f)

In [1]:
from ultralytics import YOLO

model = YOLO(r"D:/Projects/Capstone Project/models/8n_final.pt")
results = model.val(data="trashcan_inst_material.yaml", imgsz=640, batch=16, device=0)


Ultralytics 8.3.190  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,261,184 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access  (ping: 6.78.5 ms, read: 12.716.8 MB/s, size: 28.9 KB)
val: Scanning D:\Projects\Capstone Project\model switching\trash_inst_material\val\labels.cache... 1204 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1204/1204 1215681.8it/s 0.0s
val: D:\Projects\Capstone Project\model switching\trash_inst_material\val\images\vid_000143_frame0000013.jpg: 1 duplicate labels removed
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first

In [3]:
from ultralytics import YOLO
import pandas as pd

# --- Model paths and dataset ---
models = {
    "YOLOv8n_final": r"D:/Projects/Capstone Project/models/8n_final.pt",
    "YOLOv8m_final": r"D:/Projects/Capstone Project/models/8m_final.pt"
}
yaml_path = "trashcan_inst_material.yaml"

# --- Store metrics ---
results_dict = {}

for name, path in models.items():
    print(f"\n🔍 Evaluating {name} ...")
    model = YOLO(path)
    res = model.val(data=yaml_path, imgsz=640, batch=16, device=0)
    results_dict[name] = res  # Store raw results

# --- Extract per-class and overall metrics ---
rows = []
for name, res in results_dict.items():
    classes = res.names
    box = res.box
    mask = getattr(res, "mask", None)

    for i, cls in enumerate(classes):
        rows.append({
            "Model": name,
            "Class": cls,
            "Box_Precision": round(box.p[i], 3),
            "Box_Recall": round(box.r[i], 3),
            "Box_mAP50": round(box.maps[i], 3),
            "Box_mAP50-95": round(box.maps[i], 3),
            "Mask_Precision": round(mask.p[i], 3) if mask else None,
            "Mask_Recall": round(mask.r[i], 3) if mask else None,
            "Mask_mAP50": round(mask.maps[i], 3) if mask else None,
            "Mask_mAP50-95": round(mask.maps[i], 3) if mask else None
        })

# --- Create DataFrame ---
df = pd.DataFrame(rows)

# --- Compute mean performance per model ---
summary = df.groupby("Model")[["Box_Precision", "Box_Recall", "Box_mAP50", "Box_mAP50-95"]].mean().round(3)

print("\n📊 Overall Comparison:")
print(summary)

print("\n📈 Detailed Per-Class Comparison:")
print(df.head(15))



🔍 Evaluating YOLOv8n_final ...
Ultralytics 8.3.190  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,261,184 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 1.00.4 MB/s, size: 22.1 KB)
val: Scanning D:\Projects\Capstone Project\model switching\trash_inst_material\val\labels.cache... 1204 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1204/1204 600968.9it/s 0.0s
val: D:\Projects\Capstone Project\model switching\trash_inst_material\val\images\vid_000143_frame0000013.jpg: 1 duplicate labels removed
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limit